# FIG05: The human/bovine homology challenge (and trout), from the site-42 DIA-NN data.

```
FIG05: The human/bovine homology challenge (and trout), from the site-42 DIA-NN data.

Panel A - PREVALENCE: how many observed peptides are UNIQUE to one species vs SHARED between
  species, from an in-silico tryptic digest of the combined FASTA. Human and bovine share far
  more peptides than either shares with trout.
Panel B - CONSEQUENCE: observed log2(A/C) for human-unique, human+bovine-shared, and
  bovine-unique peptides. Because human (45->3 %) and cow (5->47 %) sum to a constant 50 % in
  every sample, SHARED peptides carry the combined signal and collapse toward ratio 1
  (log2 0) -- so they cannot resolve the two species and bias protein roll-up. Unique peptides
  sit at their expected ratios (human +3.9, cow -3.2).

(Panel C in the paper -- Skyline chromatograms of two single-amino-acid-variant (SAAV)
 peptides differing by one residue, showing near-identical elution -- is an instrument
 screenshot added manually; not generated here.)

Input: DIA-NN report.pr_matrix.tsv (same site-42 run as Fig 2).
Colors: Human orange, Cow green, Trout blue; human+cow-shared in purple.
```

In [1]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

DATA42 = r"D:/2022 Multi-Species Standard Study/42"
INPUT = os.path.join(DATA42, "DIANN_out", "report.pr_matrix.tsv")
FASTA = os.path.join(DATA42, "EncyclopeDIA", "Combined_proteomes.fasta")
PEPTIDE_COL = "Stripped.Sequence"
COLS = {"A": r"D:\2022 Multi-Species Standard Study\42\EncyclopeDIA\42_exploris480_DIA_A.mzML",
        "C": r"D:\2022 Multi-Species Standard Study\42\EncyclopeDIA\42_exploris480_DIA_C.mzML"}

OUTPUT = "output"
DATA = "data"
os.makedirs(OUTPUT, exist_ok=True)
os.makedirs(DATA, exist_ok=True)

COLORS = {"Human": "#E69F00", "Bovine": "#009E73", "Trout": "#56B4E9"}
SHARED_COLOR = "#CC79A7"        # human+cow shared
ORGBIT = {"Human": 1, "Bovine": 2, "Trout": 4}
# bitmask -> readable category
CAT = {1: "Human only", 2: "Bovine only", 4: "Trout only",
       3: "Human+Bovine", 5: "Human+Trout", 6: "Bovine+Trout", 7: "All three"}
EXPECTED_AC = {"Human": np.log2(45 / 3), "Bovine": np.log2(5 / 47), "Trout": 0.0}

In [2]:
def clean_pep(p):
    return re.sub(r"[^A-Z]", "", re.sub(r"\[.*?\]", "", str(p).upper()))

In [3]:
def digest(seq, missed=2, min_len=6, max_len=50):
    seq = re.sub(r"[^A-Z]", "", seq.upper())
    cuts = [0] + [i + 1 for i in range(len(seq) - 1) if seq[i] in "KR" and seq[i + 1] != "P"] + [len(seq)]
    cuts = sorted(set(cuts))
    peps = set()
    for i in range(len(cuts) - 1):
        for m in range(missed + 1):
            j = i + 1 + m
            if j >= len(cuts):
                break
            p = seq[cuts[i]:cuts[j]]
            if min_len <= len(p) <= max_len:
                peps.add(p)
    return peps

In [4]:
def fasta_org(header):
    tok = header.split()[0]
    if "_HUMAN" in tok:
        return "Human"
    if "_BOVIN" in tok:
        return "Bovine"
    return "Trout"

In [5]:
def classify(targets):
    """peptide -> organism bitmask -> category, over observed peptides. Cached."""
    cache = os.path.join(DATA, "fig5_peptide_category.csv")
    if os.path.exists(cache):
        c = pd.read_csv(cache)
        if set(targets).issubset(set(c["peptide"].astype(str))):
            return c
        print("  cache does not cover current peptides -> rebuilding")
    member = {p: 0 for p in targets}

    def flush(header, parts):
        if not header:
            return
        bit = ORGBIT[fasta_org(header)]
        for p in digest("".join(parts)):
            if p in member:
                member[p] |= bit

    header, parts = None, []
    with open(FASTA, encoding="utf-8", errors="ignore") as fh:
        for line in fh:
            if line.startswith(">"):
                flush(header, parts)
                header, parts = line[1:].strip(), []
            else:
                parts.append(line.strip())
        flush(header, parts)

    df = pd.DataFrame({"peptide": list(member), "bitmask": list(member.values())})
    df["category"] = df["bitmask"].map(lambda b: CAT.get(b, "Unassigned"))
    df.to_csv(cache, index=False)
    return df

In [6]:
def load():
    q = pd.read_csv(INPUT, sep="\t", engine="python", on_bad_lines="skip")
    q.columns = [str(c).strip() for c in q.columns]
    q["peptide"] = q[PEPTIDE_COL].map(clean_pep)
    for k, col in COLS.items():
        q[k] = pd.to_numeric(q[col], errors="coerce")
    q = q.groupby("peptide", as_index=False)[["A", "C"]].sum(min_count=1)
    cat = classify(set(q["peptide"]))
    m = q.merge(cat[["peptide", "category"]], on="peptide", how="left")
    m["log2_AC"] = np.log2(m["A"] / m["C"])
    return m

In [7]:
CAT_ORDER = ["Human only", "Bovine only", "Trout only",
             "Human+Bovine", "Human+Trout", "Bovine+Trout", "All three"]
CAT_COLOR = {"Human only": COLORS["Human"], "Bovine only": COLORS["Bovine"],
             "Trout only": COLORS["Trout"], "Human+Bovine": SHARED_COLOR,
             "Human+Trout": "#999999", "Bovine+Trout": "#999999", "All three": "#555555"}

In [8]:
def figure(m, out_png):
    fig = plt.figure(figsize=(12.5, 5.2))
    gs = fig.add_gridspec(1, 2, width_ratios=[1.05, 1], wspace=0.28)

    # Panel A: peptide sharing prevalence
    axA = fig.add_subplot(gs[0, 0])
    counts = m["category"].value_counts()
    cats = [c for c in CAT_ORDER if c in counts.index]
    vals = [counts[c] for c in cats]
    axA.bar(range(len(cats)), vals, color=[CAT_COLOR[c] for c in cats], edgecolor="black", linewidth=0.4)
    for i, v in enumerate(vals):
        axA.text(i, v * 1.03, f"{v:,}", ha="center", va="bottom", fontsize=8)
    axA.set_yscale("log")
    axA.set_xticks(range(len(cats)))
    axA.set_xticklabels(cats, rotation=30, ha="right", fontsize=9)
    axA.set_ylabel("observed peptides (log scale)")
    axA.set_title("A  Peptide sharing between species", fontsize=12, loc="left")
    axA.grid(True, axis="y", lw=0.3)

    # Panel B: quant consequence for human-unique / shared / bovine-unique
    axB = fig.add_subplot(gs[0, 1])
    groups = ["Human only", "Human+Bovine", "Bovine only"]
    gcolor = [COLORS["Human"], SHARED_COLOR, COLORS["Bovine"]]
    data = [m.loc[(m["category"] == g) & np.isfinite(m["log2_AC"]), "log2_AC"].values for g in groups]
    bp = axB.boxplot(data, orientation="horizontal", tick_labels=groups, widths=0.6,
                     showfliers=False, patch_artist=True)
    for patch, c in zip(bp["boxes"], gcolor):
        patch.set_facecolor(c); patch.set_alpha(0.55)
    for med in bp["medians"]:
        med.set_color("black")
    rng = np.random.default_rng(0)
    for i, (g, arr) in enumerate(zip(groups, data), start=1):
        if len(arr):
            xs = rng.choice(arr, size=min(len(arr), 400), replace=False)
            y = np.full(len(xs), i) + rng.uniform(-0.12, 0.12, size=len(xs))
            axB.scatter(xs, y, s=8, alpha=0.35, color=gcolor[i - 1], linewidth=0, zorder=3)
    # expected reference lines
    axB.axvline(EXPECTED_AC["Human"], ls="--", lw=1.2, color=COLORS["Human"], zorder=1)
    axB.axvline(EXPECTED_AC["Bovine"], ls="--", lw=1.2, color=COLORS["Bovine"], zorder=1)
    axB.axvline(0.0, ls="--", lw=1.2, color="black", zorder=1)
    axB.annotate("shared collapse\nto ratio 1", xy=(0, 2.0), xytext=(2.4, 2.55),
                 fontsize=8, color=SHARED_COLOR, ha="left", va="center",
                 arrowprops=dict(arrowstyle="->", color=SHARED_COLOR, lw=1))
    axB.set_xlabel("observed log2(A/C)")
    axB.set_title("B  Homology collapses shared-peptide ratios", fontsize=12, loc="left")
    axB.grid(True, axis="x", lw=0.3)
    axB.set_xlim(-6, 7)

    fig.suptitle("FIG05  Human/bovine homology challenge (site 42, DIA-NN)", fontsize=13)
    fig.tight_layout(rect=(0, 0, 1, 0.95))
    fig.savefig(out_png, dpi=200)
    fig.savefig(out_png.replace(".png", ".pdf"))
    plt.close(fig)
    print(f"  wrote {out_png}")

In [9]:
# ---- generate figure ----
m = load()
m.to_csv(os.path.join(DATA, "fig5_peptide_categories_long.csv"), index=False)
print("observed peptides by category:")
print(m["category"].value_counts().to_string())
print("\nmedian log2(A/C) by group:")
for g in ["Human only", "Human+Bovine", "Bovine only"]:
    v = m.loc[(m["category"] == g) & np.isfinite(m["log2_AC"]), "log2_AC"]
    print(f"  {g:14} median={np.median(v):+.2f}  (2^ = {2**np.median(v):.2f}), n={len(v)}")
figure(m, os.path.join(OUTPUT, "FIG05_homology_challenge.png"))

observed peptides by category:
category
Bovine only     4918
Human only      4861
Human+Bovine    4681
Trout only      2996
All three       2791
Unassigned       309
Human+Trout      161
Bovine+Trout     112

median log2(A/C) by group:
  Human only     median=+3.82  (2^ = 14.16), n=1452
  Human+Bovine   median=-0.13  (2^ = 0.92), n=3889
  Bovine only    median=-3.26  (2^ = 0.10), n=2409


C:\Users\linds\AppData\Local\Temp\ipykernel_21904\538976580.py:50: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout(rect=(0, 0, 1, 0.95))


  wrote output\FIG05_homology_challenge.png
